# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [70]:
# Escribe tu código aquí para limpiar y enriquecer los datos
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [71]:
df = pd.read_csv("../data/interim/train_set.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,NEAR BAY
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,<1H OCEAN
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,INLAND
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,INLAND
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,NEAR OCEAN


In [72]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16512 entries, 0 to 16511
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           16512 non-null  float64
 1   latitude            16512 non-null  float64
 2   housing_median_age  16512 non-null  float64
 3   total_rooms         16512 non-null  float64
 4   total_bedrooms      16344 non-null  float64
 5   population          16512 non-null  float64
 6   households          16512 non-null  float64
 7   median_income       16512 non-null  float64
 8   median_house_value  16512 non-null  float64
 9   ocean_proximity     16512 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.3 MB


1. Solventar Problemas de calidad

- Solventar problemas de sensibilidad: Existen outliers en algunas columnas, sin embargo, son datos importantes y necesarios, por lo que se mantendrán los valores.

- Solventar problemas de precicisión: En el análisis de duplicados, no existen duplicados explícitos como tampoco existen duplicados engañosos, por lo que esta sección no modificará datos.

- Solventar problemas de completitud: considerando que es MCAR, se usará MICE para llenar los datos faltantes.

In [73]:
# Solventar problemas de completitud, para mecanismo MCAR.
# Identificar filas con datos faltantes
datos_con_faltantes = df[df.isnull().any(axis=1)]
datos_con_faltantes.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
99,-120.67,40.50,15.0,5343.0,NaN,2503.0,902.0,3.5962,85900.0,INLAND
194,-117.96,34.03,35.0,2093.0,NaN,1755.0,403.0,3.4115,150400.0,<1H OCEAN
249,-118.05,34.04,33.0,1348.0,NaN,1098.0,257.0,4.2917,161200.0,<1H OCEAN
267,-118.88,34.17,15.0,4260.0,NaN,1701.0,669.0,5.1033,410700.0,<1H OCEAN
438,-117.87,33.62,8.0,1266.0,NaN,375.0,183.0,9.8020,500001.0,<1H OCEAN


In [74]:
print("Valores faltantes antes:", df["total_bedrooms"].isnull().sum())

Valores faltantes antes: 168


In [75]:
# Imputación de datos faltantes en la columna "total_bedrooms" usando MICE
imputer = IterativeImputer(max_iter=10,random_state=42)
columnas_numericas = df.select_dtypes(include="number").columns
df[columnas_numericas] = imputer.fit_transform(df[columnas_numericas])

In [76]:
# Verificación de la imputación
print("Valores faltantes después:", df["total_bedrooms"].isnull().sum())

Valores faltantes después: 0


2. Codificación Categórica

In [77]:
# verificar valores únicos en ocean_proximity
df["ocean_proximity"].unique()

<StringArray>
['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']
Length: 5, dtype: str

In [78]:
# valor promedio de median_house_value en las diferentes categorías de ocean_proximity
df.groupby("ocean_proximity")["median_house_value"].mean()

ocean_proximity
<1H OCEAN     239747.487490
INLAND        124991.118091
ISLAND        450000.000000
NEAR BAY      260416.009751
NEAR OCEAN    248372.410244
Name: median_house_value, dtype: float64

In [79]:
# ordenar de mayor a menor el valor promedio de median_house_value en las diferentes categorías de ocean_proximity
df.groupby("ocean_proximity")["median_house_value"].mean().sort_values(ascending=False)

ocean_proximity
ISLAND        450000.000000
NEAR BAY      260416.009751
NEAR OCEAN    248372.410244
<1H OCEAN     239747.487490
INLAND        124991.118091
Name: median_house_value, dtype: float64

Para la trasnformación de las variables categóricas se usa OrdinalEncoder, sin especificar un valor, de modo que la librería encuentre los valores adecuados, ya que no se pudo encontrar un patrón adecuado de clasificación.

In [80]:
#Considerando la longitud y latitud, se puede inferir que las categorías de ocean_proximity tienen un orden lógico, relacionado con la cercanía al océano. Por lo tanto, se puede usar ordinal encoding para convertir los datos de ocean_proximity a números.
encoder = OrdinalEncoder()
df["ocean_proximity"] = encoder.fit_transform(df[["ocean_proximity"]])
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,3.0
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,0.0
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,1.0
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,1.0
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,4.0


3. Enriquecimiento (Feature Engineering)

 - métricas útiles:
    - `rooms_per_household = total_rooms / households`
    - `bedrooms_per_room = total_bedrooms / total_rooms`
    - `population_per_household = population / households`

In [81]:
# Creamos 3 variables nuevas más informativas
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

print("Nuevas variables:")
df[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].describe()

Nuevas variables:


,rooms_per_household,bedrooms_per_room,population_per_household
count,16512.000000,16512.000000,16512.000000
mean,5.441010,0.212859,2.995974
std,2.574143,0.057971,4.457373
min,0.888889,-0.742015,0.692308
25%,4.443636,0.175318,2.433426
50%,5.235573,0.203201,2.822316
75%,6.053843,0.239591,3.286385
max,141.909091,1.000000,502.461538


4. Escalado de variables